In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.ticker as mticker


DATA_BASE_PATH = "../local/Tvarminne/preprocessed_all"

COLS_NON_NUMERIC = ["day", "season"]

KEY_WINTER = "winter"
KEY_SPRING = "spring"
KEY_SUMMER = "summer"
KEY_AUTUMN = "autumn"

SEASONS = [KEY_WINTER, KEY_SPRING, KEY_SUMMER, KEY_AUTUMN]

MONTH_SEASON_MAP = {
    1: SEASONS[0],
    2: SEASONS[0],
    3: SEASONS[1],
    4: SEASONS[1],
    5: SEASONS[1],
    6: SEASONS[2],
    7: SEASONS[2],
    8: SEASONS[2],
    9: SEASONS[3],
    10: SEASONS[3],
    11: SEASONS[3],
    12: SEASONS[0]
}

COMBINED_PATH = "../local/Tvarminne/combined/combined.csv"

df = pd.read_csv(COMBINED_PATH, engine='pyarrow', index_col="Time")

In [ ]:
datasets_other = {
    "full": df.copy()
}
datasets_season = {season: df[df["season"] == season] for season in SEASONS}
datasets_day = {
    "day": df[df["day"] == True],
    "night": df[df["day"] == False]
}

datasets_season_day = {f"{s} day": d_season[d_season["day"] == True] for s, d_season in datasets_season.items()}
datasets_season_night = {f"{s} night": d_season[d_season["day"] == False] for s, d_season in datasets_season.items()}


datasets = datasets_other | datasets_season | datasets_day | datasets_season_day | datasets_season_night

GROUP_MAP = {
    "Dominik" : ["CO_ppb", "NO_ppb", "SO2_ppb","SA_moleccm3","Dimers_moleccm3"],
    "Areeb" : ["salinity_psu", "TDS_mgl", "Reagents_moleccm3", "FCO2Li_umolm2s", "temperature_c"],
    "marineaurora1": ["CHL_RFU", "PE_rfu", "OD_mgl", "temperature_c", "PAR2_wm2"],
    "marineaurora2": ["CHL_RFU", "PE_rfu", "temperature_c", "WindUcomponent", "WindVcomponent", "dms_ppt", "FCO2Li_umolm2s"],
    "meteojeni1": ["WindUcomponent", "WindVcomponent", "turbidity_fnu", "CHL_RFU", "PE_rfu"],
    "meteojeni2trace": ["WindUcomponent", "WindVcomponent", "dms_ppt", ""],
}

In [ ]:
def corrfunc(x, y, **kws):
    r_pearson = float(x.corr(y, method="pearson"))
    r_spearman = float(x.corr(y, method="spearman"))

    ax = plt.gca()
    ax.annotate(f"Pearson r = {r_pearson:.2f}\nSpearman ρ = {r_spearman:.2f}",
                xy=(.5, .5),
                xycoords=ax.transAxes,
                ha='center',
                va='center',
                fontsize=12)

def scatter_reg(x, y, **kwargs):
    ax = plt.gca()
    
    # Scatter plot
    sns.scatterplot(
        x=x, y=y, ax=ax,
        alpha=1,
        size=1,
        edgecolor="none",
        **kwargs
    )
    
    # Regression line
    sns.regplot(
        x=x,
        y=y,
        ax=ax,
        scatter=False,
        line_kws={"color": "red"}
    )

    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(nbins=3)
    )

    ax.yaxis.set_major_locator(
        mticker.MaxNLocator(nbins=3)
    )

    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

def plot_pairplot(df, title):
    g = sns.PairGrid(df)
    g.map_upper(corrfunc)          # correlation in upper triangle
    g.map_lower(scatter_reg)
    # g.map_lower(sns.scatterplot)   # scatterplots
    g.map_diag(sns.histplot, bins=10)       # distributions
    plt.title(title)
    plt.show()

In [ ]:
for group_name, group_columns in GROUP_MAP.items():
    print(group_name)
    for df_name, df in datasets.items():
        df = df.drop(columns=COLS_NON_NUMERIC)
        plot_pairplot(df[group_columns], f"pairplot for {df_name} dataset")